5년동안 행복지수가 지속적으로 상승하거나 적어도 전년도와 같은 나라를 찾아 출력
하시오.

In [2]:
import os
import sys
import urllib.request
from pyspark.sql import SparkSession

# 1. VS Code 환경 에러 방지 세팅
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

spark = SparkSession.builder.appName("Q3_Simplified").master("local[*]").getOrCreate()
sc = spark.sparkContext

# 2. 데이터 다운로드
base_url = "https://raw.githubusercontent.com/nongaussian/class-2024-datamining/main/data/World%20Hapiness%20Report/"
years = [2015, 2016, 2017, 2018, 2019]

for y in years:
    urllib.request.urlretrieve(f"{base_url}{y}.csv", f"{y}.csv")

# 3. 데이터 로드 및 파싱 함수
def load_rdd(year):
    rdd = sc.textFile(f"{year}.csv")
    header = rdd.first()
    # 첫 줄(헤더)을 빼고 콤마로 나눈 뒤 (국가명, 행복지수)만 추출
    return rdd.filter(lambda line: line != header) \
              .map(lambda line: line.split(',')) \
              .map(lambda p: (p[0], float(p[2])))

# 5개년 데이터를 한 번에 리스트로 생성
rdds = [load_rdd(y) for y in years]

# 4. 연쇄 조인 및 필터링 (for문을 이용해 극도로 간결화)
joined_rdd = rdds[0] # 2015년 데이터로 시작

for next_rdd in rdds[1:]:
    # 다음 해 데이터와 조인 -> 상승/유지 필터링 -> 최신 점수(v[1])만 갱신해서 덮어씌움
    joined_rdd = joined_rdd.join(next_rdd) \
                           .filter(lambda x: x[1][0] <= x[1][1]) \
                           .mapValues(lambda v: v[1])

# 5. 국가 이름만 꺼내서 알파벳 순 정렬 후 출력
result = joined_rdd.map(lambda x: x[0]).sortBy(lambda c: c).collect()
print(result)

['Benin', 'Burkina Faso', 'Cambodia', 'Cameroon', 'Chad', 'Congo (Brazzaville)', 'Czech Republic', 'Dominican Republic', 'Estonia', 'Finland', 'Gabon', 'Honduras', 'Hungary', 'Ivory Coast', 'Latvia', 'Malta', 'Mongolia', 'Niger', 'Philippines', 'Poland', 'Portugal', 'Romania', 'Senegal', 'Serbia', 'Slovakia', 'Syria', 'Tajikistan', 'Togo']
